## Openverse underwater images pipeline

This section uses the public Openverse API to download underwater images
with metadata for ML training. No API key is required for light use.

The code will:
- Query Openverse for underwater photos via the image search API.
- Download images into `openverse_underwater/images/`.
- Write `openverse_underwater/metadata.jsonl` with prompts and metadata.
- Plot 10 random sample images with `plt.imshow`.

In [14]:
import os
import json
from pathlib import Path
from urllib.parse import urlparse

import requests
from PIL import Image
import matplotlib.pyplot as plt

# === CONFIG ===
OPENVERSE_API_ROOT = "https://api.openverse.org/v1/images"
SEARCH_QUERY = "underwater"   # adjust if you want a different concept
PER_PAGE = 20                 # number of results to request

OUT_DIR = Path("openverse_underwater")
IMG_DIR = OUT_DIR / "images"
META_PATH = OUT_DIR / "metadata.jsonl"

OUT_DIR.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)


def search_openverse(query: str, page: int = 1, per_page: int = PER_PAGE):
    resp = requests.get(
        OPENVERSE_API_ROOT,
        params={
            "q": query,
            "page": page,
            "page_size": per_page,
            "license_type": "all",   # or "commercial" if you need that
        },
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()


data = search_openverse(SEARCH_QUERY, page=1, per_page=PER_PAGE)
results = data.get("results", [])
print("Results returned:", len(results))

all_records = []

for item in results:
    url = item.get("url")
    if not url:
        continue

    title = (item.get("title") or "").strip()
    creator = item.get("creator") or ""
    source = item.get("foreign_landing_url") or ""
    license_ = item.get("license") or ""
    license_version = item.get("license_version") or ""

    rec = {
        "id": item.get("id"),
        "url": url,
        "caption": title,
        "creator": creator,
        "source_page": source,
        "license": license_,
        "license_version": license_version,
    }
    all_records.append(rec)

print("Collected Openverse records:", len(all_records))


def download_image(url: str, dest: Path) -> bool:
    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        dest.parent.mkdir(parents=True, exist_ok=True)
        with open(dest, "wb") as f:
            f.write(r.content)
        return True
    except Exception as e:
        print("Failed to download:", url, "->", e)
        return False


downloaded = []

for i, rec in enumerate(all_records, 1):
    url = rec["url"]
    parsed = urlparse(url)
    name = os.path.basename(parsed.path)
    if not name:
        name = f"openverse_{rec['id']}.jpg"

    dest = IMG_DIR / name

    if not dest.exists():
        ok = download_image(url, dest)
        if not ok:
            continue

    rel_path = dest.relative_to(OUT_DIR)
    downloaded.append({**rec, "file_name": str(rel_path)})

print("Downloaded images:", len(downloaded))

with META_PATH.open("w") as f:
    for rec in downloaded:
        text = rec["caption"] or "underwater photograph"
        obj = {
            "file_name": rec["file_name"],
            "text": text,
            "creator": rec["creator"],
            "source_page": rec["source_page"],
            "license": rec["license"],
            "license_version": rec["license_version"],
        }
        f.write(json.dumps(obj) + "\n")

print(f"Wrote {len(downloaded)} entries to {META_PATH}")

# Plot up to 10 sample images
samples = downloaded[:10]
if not samples:
    print("No images downloaded, cannot plot samples.")
else:
    n = len(samples)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    if n == 1:
        axes = [axes]
    for ax, rec in zip(axes, samples):
        img_path = OUT_DIR / rec["file_name"]
        try:
            img = Image.open(img_path)
            ax.imshow(img)
            ax.axis("off")
            ax.set_title("Openverse", fontsize=8)
        except Exception as e:
            ax.text(0.5, 0.5, "error", ha="center", va="center")
            ax.axis("off")
    plt.tight_layout()


RuntimeError: Set UNSPLASH_ACCESS_KEY env var or edit UNSPLASH_ACCESS_KEY in the notebook.